In [22]:
# import libraries
import pandas as pd
import random
import re
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import sys

In [17]:
# load the data
with open("../01_data/annotations.json", "r") as f:
    data = json.load(f)

# load the tokenizer and model and move to device
checkpoint = "HuggingFaceTB/SmolLM-1.7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [19]:
def compile_augmentation_prompt(task, tokenizer):
    sentence = task["sentence"]
    entities = task["annotations"]

    entity_texts = [e["text"] for e in entities]
    entity_list = ", ".join(entity_texts)

    # Build chat template
    if not entities:
        chat = [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant that paraphrases sentences. "
                    "Paraphrase the following sentence in one single sentence."
                ),
            },
            {
                "role": "user",
                "content": f"Sentence: {sentence}",
            },
        ]
    else:
        chat = [
            {
                "role": "system",
                "content": (
                    f"You are a helpful assistant that paraphrases sentences. "
                    f"Paraphrase the following sentence in one single sentence while keeping these entities unchanged: {entity_list}. "
                    "Use each of these entities exactly as often and in the order as they appear in this list."
                ),
            },
            {
                "role": "user",
                "content": f"Sentence: {sentence}",
            },
        ]

    prompt = tokenizer.apply_chat_template(
        chat,
        tokenize=False,
        add_generation_prompt=True,
    )

    return prompt

In [21]:
sys.path.append("../02_utils/")
from data_augmentation import augmentation_non_entity, augmentation_entity, find_entity_span

In [ ]:
# apply augmentation to certain proportion of the dataset
prop = 0.5
split_idx = int(len(data) * prop)
data_to_augment = data[split_idx:]

# empty list to store new augmented data in
augmented_dataset = []

# loop through all sentences for which augmentation should be applied
for task in data_to_augment:
    # compile the prompt and get the prompt ids
    prompt = compile_augmentation_prompt(task, tokenizer=tokenizer)
    prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    # generate the output and decode back to actual text
    outputs = model.generate(**prompt_ids)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # get only the answer text
    new_text = generated_text.split("assistant\n")[-1]
    # find the spans of the entities and append to list
    new_task = find_entity_span(task, new_text)
    augmented_dataset.append(new_task)

# concatenate both datasets and shuffle randomly
new_dataset = data + augmented_dataset
random.shuffle(new_dataset)

# export the new augmented dataset
with open("../01_data/augmented_annotations.json", "w") as f:
    json.dump(new_dataset, f)